# NAFNet Online Demo on Image Debluring

## Git clone [NAFNet](https://github.com/megvii-research/NAFNet) repo

In [1]:
!git clone https://github.com/megvii-research/NAFNet
%cd NAFNet

Cloning into 'NAFNet'...
remote: Enumerating objects: 521, done.
remote: Counting objects: 100% (339/339), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 521 (delta 263), reused 223 (delta 223), pack-reused 182 (from 1)
Receiving objects: 100% (521/521), 16.19 MiB | 21.48 MiB/s, done.
Resolving deltas: 100% (279/279), done.
/content/NAFNet


## Set up the enviroment

In [2]:
!pip install -r requirements.txt
!pip install --upgrade --no-cache-dir gdown
!python3 setup.py develop --no_cuda_ext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.9/325.9 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 32.1 MB/s eta 0:00:00
/usr/local/lib/python3.12/dist-packages/setuptools/__init__.py:94: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
!!

        ********************************************************************************
        Requirements should be satisfied by a PEP 517 installer.
        If you are using pip, you can try `pip install --use-pep517`.
        ********************************************************************************

!!
  dist.fetch_build_eggs(dist.setup_requires)
running develop
/usr/local/lib/python3.12/dist-packages/setuptools/command/develop.py:41: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

      

## Download pretrained models

In [3]:
import gdown
gdown.download('https://drive.google.com/uc?id=14D4V4raNYIOhETfcuuLI3bGLB-OYIv6X', "./experiments/pretrained_models/", quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=14D4V4raNYIOhETfcuuLI3bGLB-OYIv6X
From (redirected): https://drive.google.com/uc?id=14D4V4raNYIOhETfcuuLI3bGLB-OYIv6X&confirm=t&uuid=084a96a7-619d-4958-b5ec-f1bb184397b8
To: /content/NAFNet/experiments/pretrained_models/NAFNet-REDS-width64.pth
100%|██████████| 272M/272M [00:02<00:00, 94.4MB/s]


'./experiments/pretrained_models/NAFNet-REDS-width64.pth'

## Preparation

In [4]:
import torch

from basicsr.models import create_model
from basicsr.utils import img2tensor as _img2tensor, tensor2img, imwrite
from basicsr.utils.options import parse
import numpy as np
import cv2
import matplotlib.pyplot as plt

def imread(img_path):
  img = cv2.imread(img_path)
  img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
  return img
def img2tensor(img, bgr2rgb=False, float32=True):
    img = img.astype(np.float32) / 255.
    return _img2tensor(img, bgr2rgb=bgr2rgb, float32=float32)

def display(img1, img2):
  fig = plt.figure(figsize=(25, 10))
  ax1 = fig.add_subplot(1, 2, 1)
  plt.title('Input image', fontsize=16)
  ax1.axis('off')
  ax2 = fig.add_subplot(1, 2, 2)
  plt.title('NAFNet output', fontsize=16)
  ax2.axis('off')
  ax1.imshow(img1)
  ax2.imshow(img2)

def single_image_inference(model, img, save_path):
      model.feed_data(data={'lq': img.unsqueeze(dim=0)})

      if model.opt['val'].get('grids', False):
          model.grids()

      model.test()

      if model.opt['val'].get('grids', False):
          model.grids_inverse()

      visuals = model.get_current_visuals()
      sr_img = tensor2img([visuals['result']])
      imwrite(sr_img, save_path)


## Create Model

In [5]:
opt_path = 'options/test/REDS/NAFNet-width64.yml'
opt = parse(opt_path, is_train=False)
opt['dist'] = False
NAFNet = create_model(opt)

 load net keys <built-in method keys of dict object at 0x7fa5b94a7240>


# Inference and Show results

In [6]:
def resize_with_aspect_ratio(img, max_size=1280):
    h, w = img.shape[:2]

    # If already small enough, return as is
    if max(h, w) <= max_size:
        return img, 1.0

    # Compute scaling factor
    scale = max_size / max(h, w)
    new_w = int(w * scale)
    new_h = int(h * scale)

    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return resized, scale


In [14]:
import os

input_dir = "input"
output_dir = "output"

for fname in os.listdir(input_dir):
  input_path = os.path.join(input_dir, fname)
  output_path = os.path.join(output_dir, fname)
  if os.path.exists(output_path):
    continue

  img_input = imread(input_path)
  img_input, scale = resize_with_aspect_ratio(img_input)
  inp = img2tensor(img_input)
  single_image_inference(NAFNet, inp, output_path)
  img_output = cv2.cvtColor(imread(output_path), cv2.COLOR_BGR2RGB)

  if scale != 1.0:
      h, w = img_input.shape[:2]
      img_output = cv2.resize(img_output, (w, h), interpolation=cv2.INTER_CUBIC)

  cv2.imwrite(str(output_path), img_output)


In [15]:
import shutil
from google.colab import files

# Path to your folder
folder_path = 'output'
zip_path = 'output.zip'

shutil.make_archive('output', 'zip', folder_path)

# Download the zip
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>